In [1]:
import os
import numpy as np
from tqdm import tqdm
import copy
import matplotlib
from matplotlib.colors import Normalize
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()
from loader import load, SOURCES

In [2]:

def get_returns(rewards, not_dones):
    returns, ep_return = list(), 0

    for i, not_done in enumerate(not_dones):
        ep_return += rewards[i].item()
        if not not_done:
            returns.append(ep_return)
            ep_return = 0

    return returns

In [ ]:
# Reference scores computed from the datasets:
# random / expert score = mean undiscounted episode return of the <env>-random / <env>-expert dataset
scores = {}
for name, link in SOURCES.items():
    if "-random-" in name or "-expert-" in name:
        _, _, rewards, not_dones = load(name, link)
        scores[name.replace("-v2", "")] = np.mean(get_returns(rewards, not_dones))

HOPPER_RANDOM_SCORE = scores["hopper-random"]
HOPPER_EXPERT_SCORE = scores["hopper-expert"]
HALFCHEETAH_RANDOM_SCORE = scores["halfcheetah-random"]
HALFCHEETAH_EXPERT_SCORE = scores["halfcheetah-expert"]
WALKER_RANDOM_SCORE = scores["walker2d-random"]
WALKER_EXPERT_SCORE = scores["walker2d-expert"]
ANT_RANDOM_SCORE = scores["ant-random"]
ANT_EXPERT_SCORE = scores["ant-expert"]
print(scores)

In [ ]:
np.random.seed(42)
ind = np.arange(0, 100_000)

limits = []

N_ENVS = 4   # halfcheetah, walker2d, hopper, ant
N_TYPES = 5  # medium-replay, random, medium, expert, medium-expert

dstype, tqs, sacos, performances = [], [], [], {}

for s, (name, link) in enumerate(SOURCES.items()):

    states, actions, rewards, not_dones = load(name, link)

    print(name)

    if "medium-expert" in name:
        dstype.append("X")
    elif "replay" in name:
        dstype.append("*")
    elif "random" in name:
        dstype.append("o")
    elif "medium" in name:
        dstype.append("P")
    elif "expert" in name:
        dstype.append("s")

    d_return = np.mean(get_returns(rewards, not_dones))
    if "hopper" in name:
        tq = (d_return - HOPPER_RANDOM_SCORE) / (HOPPER_EXPERT_SCORE - HOPPER_RANDOM_SCORE)
    elif "halfcheetah" in name:
        tq = (d_return - HALFCHEETAH_RANDOM_SCORE) / (HALFCHEETAH_EXPERT_SCORE - HALFCHEETAH_RANDOM_SCORE)
    elif "walker" in name:
        tq = (d_return - WALKER_RANDOM_SCORE) / (WALKER_EXPERT_SCORE - WALKER_RANDOM_SCORE)
    elif name.startswith("ant"):
        tq = (d_return - ANT_RANDOM_SCORE) / (ANT_EXPERT_SCORE - ANT_RANDOM_SCORE)
    tqs.append(tq)

    pairs = np.concatenate((states, actions), axis=1)[ind]

    if s < N_ENVS:
        limits_ = []
        for axis in range(len(pairs[0])):
            limits_.append((np.min(pairs[:, axis]), np.max(pairs[:, axis])))
        limits.append(limits_)

    for axis in range(len(pairs[0])):
        axmin, axmax = limits[s % N_ENVS][axis]

        pairs[:, axis] = np.digitize(pairs[:, axis], np.linspace(axmin, axmax, num=5))
    pairs.astype(int)

    unique = set()
    for state in tqdm(pairs,
                      desc=f"Search for Unique States in whole dataset {name}",
                      total=len(pairs)):
        unique.add(",".join([str(s) for s in state]))
    sacos.append(len(unique))

sacos_ = copy.copy(sacos)
for i in range(N_ENVS):
    for j in range(N_TYPES):
        sacos_[i + j * N_ENVS] = sacos[i + j * N_ENVS] / sacos[i]

print(dstype)
print(tqs)
print(sacos)
print(sacos_)

load datafile: 100%|██████████| 5/5 [00:00<00:00, 79.37it/s]


halfcheetah-medium-replay-v0


Search for Unique States in whole dataset halfcheetah-medium-replay-v0: 100%|██████████| 100000/100000 [00:00<00:00, 127225.92it/s]
load datafile: 100%|██████████| 5/5 [00:00<00:00, 92.58it/s]


walker2d-medium-replay-v0


Search for Unique States in whole dataset walker2d-medium-replay-v0: 100%|██████████| 100000/100000 [00:00<00:00, 135265.40it/s]
load datafile: 100%|██████████| 5/5 [00:00<00:00, 71.42it/s]


...

walker2d-expert-v0


Search for Unique States in whole dataset walker2d-expert-v0: 100%|██████████| 100000/100000 [00:00<00:00, 133868.56it/s]
load datafile: 100%|██████████| 5/5 [00:00<00:00, 15.67it/s]


hopper-expert-v0


Search for Unique States in whole dataset hopper-expert-v0: 100%|██████████| 100000/100000 [00:00<00:00, 200800.08it/s]

['*', '*', '*', 'o', 'o', 'o', 'P', 'P', 'P', 's', 's', 's']
[0.2078731122389764, 0.07772554454841242, 0.1508000775106667, -0.0018279380033688264, -0.00015761675131240922, 0.09819066922289242, 0.34062704911011804, 0.10820650768265962, 0.32017156989216194, 1.003987811475823, 0.6323248683998477, 1.0893964661749989]
[81198, 96597, 32455, 99994, 99941, 9054, 31617, 85448, 21706, 50972, 90318, 19066]
[1.0, 1.0, 1.0, 1.231483534077194, 1.0346180523204653, 0.27897088276074566, 0.389381511859898, 0.8845823369255774, 0.6688029579417655, 0.6277494519569448, 0.9349979813037672, 0.5874595593899246]


In [5]:
sacos_ = copy.copy(sacos)
for i in range(N_ENVS):
    for j in range(N_TYPES):
        sacos_[i + j * N_ENVS] = sacos[i + j * N_ENVS] / sacos[i]